## **설정**

In [ ]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## Import libaries
import pandas as pd
import numpy as np

import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV

In [ ]:
## Setting
base_path = '/content/drive/MyDrive/2025-2 통계계산특론/우리코드/'

## **데이터 불러오기**

In [ ]:
## Load the data
file_path = os.path.join(base_path, '00_mergedData', '01_M5020_F2020')

train2018 = pd.read_csv(os.path.join(file_path, 'train2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2018 = pd.read_csv(os.path.join(file_path, 'valid2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2018 = pd.read_csv(os.path.join(file_path, 'test2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

train2019 = pd.read_csv(os.path.join(file_path, 'train2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2019 = pd.read_csv(os.path.join(file_path, 'valid2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2019 = pd.read_csv(os.path.join(file_path, 'test2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

train2020 = pd.read_csv(os.path.join(file_path, 'train2020.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2020 = pd.read_csv(os.path.join(file_path, 'valid2020.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2020 = pd.read_csv(os.path.join(file_path, 'test2020.csv')).sort_values(by=['date', 'gvkey'])

train2021 = pd.read_csv(os.path.join(file_path, 'train2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2021 = pd.read_csv(os.path.join(file_path, 'valid2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2021 = pd.read_csv(os.path.join(file_path, 'test2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

In [ ]:
train2018.head()

,ticker,gvkey,permno,sic,exchcd,shrcd,ffi49,ret,date,hs_20_0_x,...,hs_20_10_y,hs_20_11_y,hs_20_12_y,hs_20_13_y,hs_20_14_y,hs_20_15_y,hs_20_16_y,hs_20_17_y,hs_20_18_y,hs_20_19_y
0,AVX,1072,81912,3670,1.0,11.0,37,0.101395,1997-01-31,-0.063167,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
1,ABS,1240,50032,5411,1.0,11.0,43,-0.013333,1997-01-31,-0.058076,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
2,AGREA,1468,13056,2771,3.0,11.0,8,-0.002203,1997-01-31,-0.049929,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
3,ASC,1573,44652,5411,1.0,11.0,43,0.027523,1997-01-31,-0.066656,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
4,ADM,1722,10516,2070,1.0,11.0,2,-0.102273,1997-01-31,-0.058967,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069


## **데이터 전처리**

### **datetime**

In [ ]:
for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name = f'{df_type}{year}'
        # Ensure the date column is converted to datetime just once for the original dataframes
        if 'date' in globals()[df_name].columns:
            globals()[df_name]['date'] = pd.to_datetime(globals()[df_name]['date'])

### **ticker와 구분 Code 백업 및 X, y분리**

In [ ]:
## backup ticker & code and split X, y, and handle date/sasdate columns explicitly

# List of columns to always drop from X dataframes after extraction of Year/Month
# 'date' and 'sasdate' are specifically targeted here as they caused errors.
cols_to_drop_from_X = ['ret', 'date', 'sasdate']

for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name = f'{df_type}{year}'
        original_df = globals()[df_name]

        # Backup info columns before any modifications to _X
        globals()[f'{df_name}_info'] = original_df.copy()[['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'date']]

        # Create _X by copying and dropping 'ret', 'date', 'sasdate'
        temp_df_X = original_df.copy()
        for col in cols_to_drop_from_X:
            if col in temp_df_X.columns:
                temp_df_X = temp_df_X.drop(col, axis=1)
        globals()[f'{df_name}_X'] = temp_df_X

        # Create _y dataframe
        globals()[f'{df_name}_y'] = original_df.copy()[['ret']]


print("Data split into _X, _y, and _info dataframes, with 'ret', 'date', 'sasdate' removed from _X.")

Data split into _X, _y, and _info dataframes, with 'ret', 'date', 'sasdate' removed from _X.


In [ ]:
## date 컬럼에서 Year, Month 추출 후 문자열로 변환
# 'date' 컬럼은 이미 df_X에서 제거되었으므로, original_df_info에서 가져옵니다.
for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name_X = f'{df_type}{year}_X'
        df_name_info = f'{df_type}{year}_info'

        # Use the 'date' column from the _info dataframe to extract Year and Month
        info_df = globals()[df_name_info]
        current_df_X = globals()[df_name_X]

        # 날짜 컬럼 확인
        if 'date' in info_df.columns and pd.api.types.is_datetime64_any_dtype(info_df['date']):
            # [수정됨] astype(str) -> astype(str).astype('category')
            # 숫자(2018) -> 문자("2018") -> 카테고리("2018") 순서로 변환해야 안전합니다.
            current_df_X['Year'] = info_df['date'].dt.year.astype(str).astype('category')
            current_df_X['Month'] = info_df['date'].dt.month.astype(str).astype('category')
        else:
            print(f"Warning: 'date' column issue in {df_name_info}")

        # 변경된 데이터프레임 재할당
        globals()[df_name_X] = current_df_X


print("Year and Month extracted and converted to 'category' type.")

Year and Month extracted and converted to 'category' type.


In [ ]:
## 2. 나머지 카테고리 변수들도 'category' 타입으로 일괄 변환
# 기존 코드의 categorical_features_final 리스트 활용
categorical_features_final = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'Year', 'Month']

# 전역 변수로 categorical_features 설정 (학습 코드에서 이 이름으로 참조함)
categorical_features = categorical_features_final

for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name_X = f'{df_type}{year}_X'
        current_df_X = globals()[df_name_X]

        for col in categorical_features:
            if col in current_df_X.columns:
                # [수정됨] astype(str) -> astype('category')
                # 이미 카테고리인 Year/Month는 유지되고, 나머지는 변환됩니다.
                current_df_X[col] = current_df_X[col].astype('category')

        globals()[df_name_X] = current_df_X

print("All designated categorical features converted to 'category' type.")

All designated categorical features converted to 'category' type.


## **공통 함수**

In [ ]:
def create_blocked_time_series_splits(df, n_splits=3):

    # 1. Year와 Month를 결합하여 절대적인 시간 순서 키 생성 (예: 201801, 201802)
    # df는 원본을 건드리지 않기 위해 내부적으로만 사용할 시리즈를 만듭니다.
    time_series_id = df['Year'] * 100 + df['Month']

    unique_times = time_series_id.unique()
    unique_times.sort() # 201801 -> 201802 -> ... -> 201901 순서로 정렬됨

    n_times = len(unique_times)

    # 데이터 기간이 너무 짧아서(예: 3개월 미만) 분할이 불가능한 경우 예외 처리
    if n_times < n_splits + 1:
        return []

    # 분할 크기 계산
    fold_size = n_times // (n_splits + 1)

    cv_indices = []

    for i in range(1, n_splits + 1):
        # 학습에 사용할 마지막 시점(인덱스) 계산
        train_end_idx = i * fold_size

        # 현재 split에서 학습용으로 쓸 'YYYYMM' 리스트
        train_times = unique_times[:train_end_idx+1]

        # 검증용으로 쓸 'YYYYMM' (학습 바로 다음 시점)
        # 여기서는 바로 다음 1개 시점(1달)을 검증으로 씁니다.
        test_time_idx = train_end_idx + 1

        if test_time_idx >= n_times:
            break

        test_time = unique_times[test_time_idx]

        # 2. 마스킹을 통해 원본 데이터프레임의 인덱스 추출
        train_mask = time_series_id.isin(train_times)
        test_mask = time_series_id == test_time

        train_idx = np.where(train_mask)[0]
        test_idx = np.where(test_mask)[0]

        cv_indices.append((train_idx, test_idx))

    return cv_indices

## **XGBoost**

In [ ]:
from xgboost import XGBRegressor

In [ ]:
categorical_features = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'Year', 'Month']

In [ ]:
xgb_2018 = XGBRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)
xgb_2019 = XGBRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)
xgb_2020 = XGBRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)
xgb_2021 = XGBRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)

In [ ]:
enable_categorical=True

In [ ]:
import xgboost as xgb
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

def tunning_xgb(
    df_train_X, df_train_y,
    df_valid_X, df_valid_y,
    cat_features=None
):
    """
    Manual grid search for XGBoost using training/validation split (no CV).
    Uses sklearn-style parameter names in grid but maps to native XGBoost params.
    """

    print("Starting XGBoost manual grid search...")

    # 1. drop 'date' if exists
    for df in [df_train_X, df_valid_X]:
        if 'date' in df.columns:
            df.drop(columns=['date'], inplace=True)

    # 2. Categorical 처리
    if cat_features:
        for col in cat_features:
            if col in df_train_X.columns and col in df_valid_X.columns:
                all_categories = pd.concat([
                    df_train_X[col].astype(str),
                    df_valid_X[col].astype(str)
                ]).unique()
                categorical_dtype = pd.CategoricalDtype(categories=all_categories, ordered=False)

                df_train_X[col] = df_train_X[col].astype(categorical_dtype)
                df_valid_X[col] = df_valid_X[col].astype(categorical_dtype)
            elif col in df_train_X.columns:
                df_train_X[col] = df_train_X[col].astype('category')
            elif col in df_valid_X.columns:
                df_valid_X[col] = df_valid_X[col].astype('category')

    # Convert to DMatrix
    dtrain = xgb.DMatrix(df_train_X, label=df_train_y, enable_categorical=True)
    dvalid = xgb.DMatrix(df_valid_X, label=df_valid_y, enable_categorical=True)

    # 3. Hyperparameter grid (요청하신 파라미터 이름 적용)
    param_grid = {
        'n_estimators': [500],        # -> num_boost_round
        'learning_rate': [0.03, 0.1], # -> eta
        'max_depth': [6, 8, 10],
        'reg_lambda': [1, 3]          # -> lambda
    }

    best_rmse = float('inf')
    best_params = None
    best_model = None
    all_results = []

    print("Starting XGBoost tuning (DMatrix mode)...")

    for params in ParameterGrid(param_grid):
        print(f"→ Params: {params}")

        # 파라미터 매핑 (sklearn style -> native xgboost style)
        xgb_params = {
            'objective': 'reg:squarederror',
            'eta': params['learning_rate'],      # learning_rate 매핑
            'max_depth': params['max_depth'],
            'lambda': params['reg_lambda'],      # reg_lambda 매핑
            'tree_method': 'hist',
        }

        watchlist = [(dtrain, 'train'), (dvalid, 'eval')]

        model = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=params['n_estimators'], # n_estimators 매핑
            evals=watchlist,
            early_stopping_rounds=50,
            verbose_eval=False
        )

        # predict using best_iteration
        pred = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))

        # Metrics
        rmse = np.sqrt(mean_squared_error(df_valid_y, pred))
        r2 = r2_score(df_valid_y, pred)

        print(f"   RMSE={rmse:.6f},  R²={r2:.6f},  best_iter={model.best_iteration}")

        all_results.append({
            'params': params,
            'rmse': rmse,
            'r2': r2,
            'best_iteration': model.best_iteration
        })

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = params
            best_model = model

    result_summary = {
        'best_params': best_params,
        'best_rmse': best_rmse,
        'all_results': all_results,
        'best_model': best_model
    }

    return best_model, result_summary

In [ ]:
print("--- 2018 Training (XGBoost) ---")
xgb_2018, result_2018 = tunning_xgb(train2018_X, train2018_y, valid2018_X, valid2018_y, cat_features=categorical_features)

print("--- 2019 Training (XGBoost) ---")
xgb_2019, result_2019 = tunning_xgb(train2019_X, train2019_y, valid2019_X, valid2019_y, cat_features=categorical_features)

print("--- 2020 Training (XGBoost) ---")
xgb_2020, result_2020 = tunning_xgb(train2020_X, train2020_y, valid2020_X, valid2020_y, cat_features=categorical_features)

print("--- 2021 Training (XGBoost) ---")
xgb_2021, result_2021 = tunning_xgb(train2021_X, train2021_y, valid2021_X, valid2021_y, cat_features=categorical_features)

--- 2018 Training (XGBoost) ---
Starting XGBoost manual grid search...
Starting XGBoost tuning (DMatrix mode)...
→ Params: {'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 1}
   RMSE=0.083891,  R²=0.000856,  best_iter=1
→ Params: {'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 3}
   RMSE=0.083886,  R²=0.000982,  best_iter=1
→ Params: {'learning_rate': 0.03, 'max_depth': 8, 'n_estimators': 500, 'reg_lambda': 1}
   RMSE=0.083922,  R²=0.000116,  best_iter=0
→ Params: {'learning_rate': 0.03, 'max_depth': 8, 'n_estimators': 500, 'reg_lambda': 3}
   RMSE=0.083908,  R²=0.000459,  best_iter=1
→ Params: {'learning_rate': 0.03, 'max_depth': 10, 'n_estimators': 500, 'reg_lambda': 1}
   RMSE=0.083899,  R²=0.000667,  best_iter=0
→ Params: {'learning_rate': 0.03, 'max_depth': 10, 'n_estimators': 500, 'reg_lambda': 3}
   RMSE=0.083942,  R²=-0.000345,  best_iter=0
→ Params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 1}

### **최종 학습**

In [ ]:
import pandas as pd
import xgboost as xgb

# Dictionary to store the CategoricalDtype objects for each model's training data
model_column_cat_dtypess = {}

for year in range(2018, 2022):
    print(f"--- Final Training for {year} ---")

    # Train + Valid 데이터 합치기
    combined_train_X = pd.concat([globals()[f'train{year}_X'], globals()[f'valid{year}_X']], ignore_index=True)
    combined_train_y = pd.concat([globals()[f'train{year}_y'], globals()[f'valid{year}_y']], ignore_index=True)

    if 'date' in combined_train_X.columns:
        combined_train_X.drop(columns=['date'], inplace=True)

    # Store CategoricalDtype for each relevant column
    current_model_cat_dtypes = {}
    if 'categorical_features' in globals() and categorical_features:
        for col in categorical_features:
            if col in combined_train_X.columns:
                # Ensure it's a string type before converting to category
                combined_train_X[col] = combined_train_X[col].astype(str)
                # Convert to category. This creates the CategoricalDtype implicitly.
                combined_train_X[col] = combined_train_X[col].astype('category')
                # Store the CategoricalDtype object
                current_model_cat_dtypes[col] = combined_train_X[col].dtype
    model_column_cat_dtypess[year] = current_model_cat_dtypes

    # DMatrix 생성 (최종 학습용 데이터)
    dtrain_final = xgb.DMatrix(combined_train_X, label=combined_train_y, enable_categorical=True)

    # 해당 연도의 최적 파라미터 가져오기
    result_dict_name = f'result_{year}'
    best_params = globals()[result_dict_name]['best_params']

    # 파라미터 매핑 (Sklearn Style -> XGBoost Native Style)
    final_xgb_params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',  # Categorical 지원을 위해 hist 사용
        'eta': best_params['learning_rate'],      # learning_rate -> eta
        'max_depth': best_params['max_depth'],
        'lambda': best_params['reg_lambda'],      # reg_lambda -> lambda
    }

    # n_estimators는 num_boost_round 인자로 따로 뺨
    num_rounds = best_params['n_estimators']

    # 최종 모델 학습 (Validation set 없이 전체 데이터로 학습)
    # Early Stopping 없이 정해진 num_rounds만큼 끝까지 학습
    final_model = xgb.train(
        params=final_xgb_params,
        dtrain=dtrain_final,
        num_boost_round=num_rounds,
        verbose_eval=False
    )

    # 학습된 모델 저장
    globals()[f'xgb_{year}'] = final_model

    print(f"Final model for {year} trained with best parameters: {best_params}")

print("All final models trained successfully on combined train+valid data.")

--- Final Training for 2018 ---
Final model for 2018 trained with best parameters: {'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 3}
--- Final Training for 2019 ---
Final model for 2019 trained with best parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 3}
--- Final Training for 2020 ---
Final model for 2020 trained with best parameters: {'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 1}
--- Final Training for 2021 ---
Final model for 2021 trained with best parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 500, 'reg_lambda': 1}
All final models trained successfully on combined train+valid data.


## **예측**

In [ ]:
def make_result_table(model, test_info, test_X, test_y):
    # 1. 원본 데이터 보존을 위해 복사
    X_temp = test_X.copy()

    # 2. 'date' 컬럼 제거
    if 'date' in X_temp.columns:
        X_temp.drop(columns=['date'], inplace=True)

    # 3. Categorical Feature 처리
    # (학습 때와 동일하게 'category' 타입으로 변환해야 DMatrix가 인식함)
    # 전역 변수 categorical_features가 있다고 가정
    if 'categorical_features' in globals() and categorical_features:
        for col in categorical_features:
            if col in X_temp.columns:
                X_temp[col] = X_temp[col].astype('category')

    # 4. DMatrix 생성 (XGBoost Native 예측용)
    dtest = xgb.DMatrix(X_temp, label=test_y, enable_categorical=True)

    # 5. 예측 수행
    pred_y = model.predict(dtest)

    # 6. 결과 DataFrame 생성
    result_df = test_info[['ticker', 'date']].copy()
    result_df['pred_ret'] = pred_y
    result_df['true_ret'] = test_y

    return result_df

In [ ]:
def make_result_table(model, test_info, test_X, test_y, training_year):
    # 1. 원본 데이터 보존을 위해 복사
    X_temp = test_X.copy()

    # 2. 'date' 컬럼 제거
    if 'date' in X_temp.columns:
        X_temp.drop(columns=['date'], inplace=True)

    # 3. Categorical Feature 처리
    # 현재 모델의 훈련 연도에 해당하는 CategoricalDtype 객체들을 가져옴
    trained_cat_dtypes = globals()['model_column_cat_dtypess'].get(training_year, {})

    if 'categorical_features' in globals() and categorical_features:
        for col in categorical_features:
            if col in X_temp.columns:
                # CategoricalDtype을 적용하기 전에 문자열 타입으로 변환
                X_temp[col] = X_temp[col].astype(str)

                # 'Year' 컬럼에 대한 특별 처리: 훈련 데이터에 없는 미래 연도를 가장 마지막 훈련 연도로 대체
                if col == 'Year' and col in trained_cat_dtypes:
                    train_years_categories = trained_cat_dtypes[col].categories
                    if not train_years_categories.empty:
                        # 훈련 데이터에서 가장 큰 연도를 찾음
                        max_train_year = train_years_categories.max()
                        # 테스트 데이터의 연도가 가장 큰 훈련 연도보다 크면, 해당 연도를 가장 큰 훈련 연도로 대체
                        X_temp[col] = X_temp[col].apply(
                            lambda x: str(max_train_year) if int(x) > int(max_train_year) else x
                        )
                    else:
                        print(f"Warning: No year categories found for {col} in training data for year {training_year}.")

                # 저장된 CategoricalDtype을 적용
                # test_X[col]의 값이 trained_cat_dtypes[col].categories에 없으면 NaN으로 변환됨 (XGBoost에서 처리 가능)
                if col in trained_cat_dtypes:
                    X_temp[col] = X_temp[col].astype(trained_cat_dtypes[col])
                else:
                    # 만약 특정 컬럼에 대해 dtype이 저장되지 않았다면 일반 category로 변환 (발생하지 않아야 함)
                    X_temp[col] = X_temp[col].astype('category')

    # 4. DMatrix 생성
    dtest = xgb.DMatrix(X_temp, label=test_y, enable_categorical=True)

    # 5. 예측 수행
    pred_y = model.predict(dtest)

    # --- [추가됨] 6. 성능 지표(RMSE, R2) 계산 및 출력 ---
    rmse = np.sqrt(mean_squared_error(test_y, pred_y))
    r2 = r2_score(test_y, pred_y)

    print(f"[{training_year}] Performance Metrics:")
    print(f"  - RMSE : {rmse:.6f}")
    print(f"  - R²   : {r2:.6f}")
    print("-" * 30)
    # -----------------------------------------------------

    # 7. 결과 DataFrame 생성
    result_df = test_info[['ticker', 'date']].copy()
    result_df['pred_ret'] = pred_y
    result_df['true_ret'] = test_y

    return result_df

In [ ]:
result_2018 = make_result_table(xgb_2018, test2018_info, test2018_X, test2018_y, training_year=2018)
result_2019 = make_result_table(xgb_2019, test2019_info, test2019_X, test2019_y, training_year=2019)
result_2020 = make_result_table(xgb_2020, test2020_info, test2020_X, test2020_y, training_year=2020)
result_2021 = make_result_table(xgb_2021, test2021_info, test2021_X, test2021_y, training_year=2021)

[2018] Performance Metrics:
  - RMSE : 0.095193
  - R²   : -0.130660
------------------------------
[2019] Performance Metrics:
  - RMSE : 0.121358
  - R²   : -0.708061
------------------------------
[2020] Performance Metrics:
  - RMSE : 0.154064
  - R²   : -0.115522
------------------------------
[2021] Performance Metrics:
  - RMSE : 0.204049
  - R²   : -0.158655
------------------------------


In [ ]:
save_dir = os.path.join(base_path, '00_predData', '01_XGB_M5020_F2020')

In [ ]:
result_2018.to_csv(os.path.join(save_dir, 'result_2018.csv'), index=False)
result_2019.to_csv(os.path.join(save_dir, 'result_2019.csv'), index=False)
result_2020.to_csv(os.path.join(save_dir, 'result_2020.csv'), index=False)
result_2021.to_csv(os.path.join(save_dir, 'result_2021.csv'), index=False)